### Customer Churn Prediction — Phase 4: Business Insights & Revenue Framing
### Telco Customer Churn Dataset
 
Steps:
  1. Load predictions + SHAP values
  2. Risk tier segmentation (High / Medium / Low)
  3. Tier profiles — who is in each bucket?
  4. Threshold tuning with business cost framing
  5. Revenue retained by the model
  6. Per-customer intervention dashboard
  7. CV-ready summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, precision_score,
                             recall_score, f1_score)
import shap
import warnings
warnings.filterwarnings("ignore")
 
plt.rcParams.update({"figure.facecolor": "white", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e5e5e5"})
 
SEED = 42
 
# ── Business assumptions ─────────────────────────────────────────────────
AVG_MONTHLY_REVENUE   = 65.0    # £ average monthly charge per customer
AVG_TENURE_MONTHS     = 32.0    # average customer lifetime
AVG_CLV               = AVG_MONTHLY_REVENUE * AVG_TENURE_MONTHS  # £2,080
RETENTION_OFFER_COST  = 50.0    # £ cost of a retention offer (discount/gift)
RETENTION_SUCCESS_RATE = 0.40  

### LOAD DATA

In [ ]:
test_df  = pd.read_csv("churn_test_predictions.csv")
train_df = pd.read_csv("churn_train_preprocessed.csv")
 
PRED_COLS    = ["y_true", "y_prob", "y_pred"]
y_test       = test_df["y_true"].values
y_prob       = test_df["y_prob"].values
y_pred       = test_df["y_pred"].values
X_test_proc  = test_df.drop(columns=PRED_COLS).values
X_train_proc = train_df.values
feature_names = test_df.drop(columns=PRED_COLS).columns.tolist()
 
# Load raw test features for readable profiles
df_raw = pd.read_csv("telco_clean.csv")
df_raw = df_raw.drop(columns=["customerID","TenureBand"], errors="ignore")
y_full = (df_raw["Churn"] == "Yes").astype(int)
_, X_test_raw, _, y_test_raw = train_test_split(
    df_raw.drop(columns=["Churn","ChurnBinary"], errors="ignore"),
    y_full, test_size=0.2, stratify=y_full, random_state=SEED
)
X_test_raw = X_test_raw.reset_index(drop=True)
 
# Retrain XGBoost for SHAP
_, _, y_train, _ = train_test_split(
    df_raw, y_full, test_size=0.2, stratify=y_full, random_state=SEED
)
xgb_model = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    random_state=SEED, eval_metric="logloss", verbosity=0,
)
xgb_model.fit(X_train_proc, y_train)
 
# SHAP values
explainer  = shap.TreeExplainer(xgb_model)
shap_vals  = explainer(X_test_proc)
shap_array = shap_vals.values
shap_df    = pd.DataFrame(shap_array, columns=feature_names)
 
print("=" * 65)
print("PHASE 4 — BUSINESS INSIGHTS & REVENUE FRAMING")
print("=" * 65)
print(f"Test customers : {len(y_test):,}")
print(f"Churn rate     : {y_test.mean()*100:.1f}%")
print(f"\nBusiness assumptions:")
print(f"  Avg monthly revenue/customer : £{AVG_MONTHLY_REVENUE:.0f}")
print(f"  Avg customer lifetime        : {AVG_TENURE_MONTHS:.0f} months")
print(f"  Avg customer lifetime value  : £{AVG_CLV:,.0f}")
print(f"  Retention offer cost         : £{RETENTION_OFFER_COST:.0f}")
print(f"  Retention success rate       : {RETENTION_SUCCESS_RATE*100:.0f}%")

### RISK TIER SEGMENTATION

In [ ]:
def assign_tier(prob):
    if prob >= 0.70:   return "High Risk"
    elif prob >= 0.40: return "Medium Risk"
    else:              return "Low Risk"
 
tiers       = np.array([assign_tier(p) for p in y_prob])
tier_order  = ["High Risk", "Medium Risk", "Low Risk"]
tier_colors = {"High Risk": "#E07070", "Medium Risk": "#FF7F0E",
               "Low Risk" : "#2ca02c"}
 
tier_summary = []
for tier in tier_order:
    mask      = tiers == tier
    n         = mask.sum()
    actual_cr = y_test[mask].mean() * 100 if n > 0 else 0
    avg_prob  = y_prob[mask].mean() * 100 if n > 0 else 0
    tier_summary.append({
        "Tier"           : tier,
        "Customers"      : n,
        "% of test set"  : round(n/len(y_test)*100, 1),
        "Actual churn %" : round(actual_cr, 1),
        "Avg pred prob %" : round(avg_prob, 1),
    })
 
tier_df = pd.DataFrame(tier_summary)
print(f"\n{'='*65}")
print("RISK TIER SEGMENTATION")
print(f"{'='*65}")
print(tier_df.to_string(index=False))
 
# ── Tier visualisation ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Risk Tier Segmentation", fontweight="bold", fontsize=13)
 
# Panel 1: Customer count per tier
ax = axes[0]
counts = [tier_df.loc[tier_df["Tier"]==t, "Customers"].values[0]
          for t in tier_order]
bars = ax.bar(tier_order, counts,
              color=[tier_colors[t] for t in tier_order],
              width=0.5, edgecolor="white")
ax.set_ylabel("Customer count")
ax.set_title("Customers per tier")
for bar, v in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+5,
            f"{v:,}", ha="center", fontsize=11)
 
# Panel 2: Actual churn rate per tier
ax2 = axes[1]
actual_crs = [tier_df.loc[tier_df["Tier"]==t, "Actual churn %"].values[0]
              for t in tier_order]
bars2 = ax2.bar(tier_order, actual_crs,
                color=[tier_colors[t] for t in tier_order],
                width=0.5, edgecolor="white")
ax2.axhline(y_test.mean()*100, color="#555", linestyle="--",
            linewidth=1, label=f"Overall churn rate ({y_test.mean()*100:.1f}%)")
ax2.set_ylabel("Actual churn rate (%)")
ax2.set_title("Actual churn rate per tier\n(model correctly separates risk)")
ax2.legend(fontsize=9)
for bar, v in zip(bars2, actual_crs):
    ax2.text(bar.get_x()+bar.get_width()/2,
             bar.get_height()+0.5,
             f"{v:.1f}%", ha="center", fontsize=11, fontweight="bold")
 
# Panel 3: Distribution of predicted probabilities
ax3 = axes[2]
ax3.hist(y_prob, bins=40, color="#4C72B0",
         edgecolor="white", alpha=0.8, label="All customers")
ax3.axvline(0.40, color="#FF7F0E", linestyle="--",
            linewidth=2, label="Medium threshold (0.40)")
ax3.axvline(0.70, color="#E07070", linestyle="--",
            linewidth=2, label="High threshold (0.70)")
ax3.set_xlabel("Predicted churn probability")
ax3.set_ylabel("Count")
ax3.set_title("Probability distribution\nwith tier boundaries")
ax3.legend(fontsize=8)
 
plt.tight_layout()
plt.savefig("phase4_risk_tiers.png", dpi=150, bbox_inches="tight")
plt.show()

### THRESHOLD TUNING — BUSINESS COST FRAMING

#    A false negative (missed churner) = lost CLV = £2,080
#    A false positive (wrong alert)    = wasted retention offer = £50
#    At each threshold: total cost = FN×CLV_LOST + FP×OFFER_COST
# ════════════════════════════════════════════════════════════════════════════
 
thresh_range = np.arange(0.10, 0.91, 0.05)
thresh_rows  = []
 
for t in thresh_range:
    y_pred_t     = (y_prob >= t).astype(int)
    cm           = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm.ravel()
 
    # Business cost at this threshold
    clv_lost     = fn * AVG_CLV * (1 - RETENTION_SUCCESS_RATE)
    offer_cost   = fp * RETENTION_OFFER_COST
    revenue_saved = tp * AVG_CLV * RETENTION_SUCCESS_RATE
    net_benefit  = revenue_saved - offer_cost - clv_lost
 
    thresh_rows.append({
        "Threshold"    : round(t, 2),
        "Precision"    : round(precision_score(y_test, y_pred_t, zero_division=0), 3),
        "Recall"       : round(recall_score(y_test, y_pred_t, zero_division=0), 3),
        "F1"           : round(f1_score(y_test, y_pred_t, zero_division=0), 3),
        "FN"           : int(fn),
        "FP"           : int(fp),
        "CLV Lost (£)" : round(clv_lost, 0),
        "Offer Cost (£)": round(offer_cost, 0),
        "Revenue Saved (£)": round(revenue_saved, 0),
        "Net Benefit (£)"  : round(net_benefit, 0),
    })
 
thresh_df     = pd.DataFrame(thresh_rows)
best_f1_idx   = thresh_df["F1"].idxmax()
best_net_idx  = thresh_df["Net Benefit (£)"].idxmax()
 
print(f"\n{'='*65}")
print("THRESHOLD TUNING — METRICS + BUSINESS COST")
print(f"{'='*65}")
print(thresh_df[["Threshold","Precision","Recall","F1",
                 "FN","FP","Net Benefit (£)"]].to_string(index=False))
 
print(f"\n  Best F1 at threshold    : "
      f"{thresh_df.loc[best_f1_idx,'Threshold']:.2f}  "
      f"(F1={thresh_df.loc[best_f1_idx,'F1']:.3f}  "
      f"Net=£{thresh_df.loc[best_f1_idx,'Net Benefit (£)']:,.0f})")
print(f"  Best net benefit at     : "
      f"{thresh_df.loc[best_net_idx,'Threshold']:.2f}  "
      f"(F1={thresh_df.loc[best_net_idx,'F1']:.3f}  "
      f"Net=£{thresh_df.loc[best_net_idx,'Net Benefit (£)']:,.0f})")
 
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Threshold Tuning — Metrics + Business Cost",
             fontweight="bold", fontsize=13)
 
# Panel 1: Precision / Recall / F1
ax = axes[0, 0]
ax.plot(thresh_df["Threshold"], thresh_df["Precision"],
        color="#4C72B0", linewidth=2, label="Precision")
ax.plot(thresh_df["Threshold"], thresh_df["Recall"],
        color="#E07070", linewidth=2, label="Recall")
ax.plot(thresh_df["Threshold"], thresh_df["F1"],
        color="#2ca02c", linewidth=2, linestyle="--", label="F1")
ax.axvline(thresh_df.loc[best_f1_idx,"Threshold"],
           color="#2ca02c", linestyle=":", linewidth=1.5,
           label=f"Best F1 (t={thresh_df.loc[best_f1_idx,'Threshold']:.2f})")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1"); ax.legend(fontsize=9)
 
# Panel 2: FN and FP counts
ax2 = axes[0, 1]
ax2.plot(thresh_df["Threshold"], thresh_df["FN"],
         color="#E07070", linewidth=2, marker="o",
         markersize=4, label="FN — churners missed")
ax2.plot(thresh_df["Threshold"], thresh_df["FP"],
         color="#4C72B0", linewidth=2, marker="s",
         markersize=4, label="FP — false alerts")
ax2.set_xlabel("Threshold"); ax2.set_ylabel("Count")
ax2.set_title("Error counts vs threshold"); ax2.legend(fontsize=9)
 
# Panel 3: Business costs
ax3 = axes[1, 0]
ax3.plot(thresh_df["Threshold"], thresh_df["CLV Lost (£)"],
         color="#E07070", linewidth=2, label="CLV lost (FN × £2,080 × 60%)")
ax3.plot(thresh_df["Threshold"], thresh_df["Offer Cost (£)"],
         color="#4C72B0", linewidth=2, label=f"Offer cost (FP × £{RETENTION_OFFER_COST:.0f})")
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"£{x/1000:.0f}k"))
ax3.set_xlabel("Threshold"); ax3.set_ylabel("Cost (£)")
ax3.set_title("Business costs vs threshold"); ax3.legend(fontsize=9)
 
# Panel 4: Net benefit
ax4 = axes[1, 1]
net_vals = thresh_df["Net Benefit (£)"].values
colors_net = ["#2ca02c" if v > 0 else "#E07070" for v in net_vals]
ax4.bar(thresh_df["Threshold"], net_vals, color=colors_net,
        width=0.04, edgecolor="none")
ax4.axhline(0, color="#333", linewidth=0.8)
ax4.axvline(thresh_df.loc[best_net_idx,"Threshold"],
            color="#2ca02c", linestyle="--", linewidth=2,
            label=f"Best net benefit (t={thresh_df.loc[best_net_idx,'Threshold']:.2f})")
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"£{x/1000:.0f}k"))
ax4.set_xlabel("Threshold"); ax4.set_ylabel("Net benefit (£)")
ax4.set_title("Net revenue benefit vs threshold\n"
              "= revenue saved − offer cost − CLV lost")
ax4.legend(fontsize=9)
 
plt.tight_layout()
plt.savefig("phase4_threshold_business.png", dpi=150, bbox_inches="tight")
plt.show()

### REVENUE FRAMING — MODEL VS NO MODEL

In [ ]:
opt_t = thresh_df.loc[best_net_idx, "Threshold"]
y_pred_opt = (y_prob >= opt_t).astype(int)
cm_opt     = confusion_matrix(y_test, y_pred_opt)
tn_o, fp_o, fn_o, tp_o = cm_opt.ravel()
 
# No model baseline: do nothing — all churners are lost
total_churners    = y_test.sum()
revenue_no_model  = 0
clv_lost_no_model = total_churners * AVG_CLV
 
# With model at optimal threshold
revenue_with_model = tp_o * AVG_CLV * RETENTION_SUCCESS_RATE
cost_with_model    = (tp_o + fp_o) * RETENTION_OFFER_COST
net_with_model     = revenue_with_model - cost_with_model
 
print(f"\n{'='*65}")
print("REVENUE IMPACT — MODEL vs NO MODEL")
print(f"{'='*65}")
print(f"\n  Scenario: {len(y_test):,} customers  |  "
      f"{total_churners} actual churners\n")
print(f"  ── No model (do nothing) ─────────────────────────────")
print(f"     Churners retained    : 0")
print(f"     Revenue saved        : £0")
print(f"     CLV lost to churn    : £{clv_lost_no_model:,.0f}")
print(f"\n  ── With model (threshold={opt_t:.2f}) ──────────────────")
print(f"     Churners flagged     : {tp_o + fp_o}")
print(f"     True churners caught : {tp_o}")
print(f"     Churners retained    : {int(tp_o * RETENTION_SUCCESS_RATE)}")
print(f"     Revenue saved        : £{revenue_with_model:,.0f}")
print(f"     Retention offer cost : £{cost_with_model:,.0f}")
print(f"     Net benefit          : £{net_with_model:,.0f}")
print(f"\n  ── Model ROI ─────────────────────────────────────────")
roi = (net_with_model / cost_with_model * 100) if cost_with_model > 0 else 0
print(f"     ROI                  : {roi:.0f}%  "
      f"(£{net_with_model:,.0f} returned per £{cost_with_model:,.0f} spent)")
 
# Comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Revenue Impact — Model vs No Model", fontweight="bold")
 
scenarios = ["No model", f"With model\n(t={opt_t:.2f})"]
rev_saved = [0, revenue_with_model]
costs     = [0, cost_with_model]
net_ben   = [0, net_with_model]
 
ax = axes[0]
x  = np.arange(2)
w  = 0.3
ax.bar(x - w/2, rev_saved, w, color="#2ca02c", label="Revenue saved", edgecolor="white")
ax.bar(x + w/2, costs,     w, color="#E07070", label="Offer cost",    edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(scenarios)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1000:.0f}k"))
ax.set_title("Revenue saved vs cost of intervention")
ax.legend(fontsize=9)
 
ax2 = axes[1]
bar_colors = ["#aaaaaa", "#2ca02c"]
bars2 = ax2.bar(scenarios, net_ben, color=bar_colors,
                width=0.4, edgecolor="white")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1000:.0f}k"))
ax2.set_title("Net benefit per month\n(revenue saved minus costs)")
for bar, v in zip(bars2, net_ben):
    ax2.text(bar.get_x()+bar.get_width()/2,
             bar.get_height() + max(net_ben)*0.02,
             f"£{v:,.0f}", ha="center", fontsize=11, fontweight="bold")
 
plt.tight_layout()
plt.savefig("phase4_revenue_impact.png", dpi=150, bbox_inches="tight")
plt.show()

### FULL INTERVENTION DASHBOARD — ALL HIGH-RISK CUSTOMERS

In [ ]:
ACTION_MAP = {
    "tenure"                       : "Early loyalty reward — flag for 3-month retention call",
    "MonthlyCharges"               : "Offer loyalty discount or downgrade option",
    "TotalCharges"                 : "Review billing history — proactive support call",
    "Contract_Two year"            : "Promote annual or two-year contract upgrade",
    "Contract_One year"            : "Upgrade to two-year contract with incentive",
    "InternetService_Fiber optic"  : "Check service quality — fibre churn risk",
    "OnlineSecurity_Yes"           : "Promote security add-on bundle",
    "TechSupport_Yes"              : "Promote tech support plan",
    "PaperlessBilling"             : "Review billing communication preference",
    "SeniorCitizen"                : "Dedicated senior support outreach",
}
 
high_risk_mask = y_prob >= 0.70
hr_indices     = np.where(high_risk_mask)[0]
hr_sorted      = hr_indices[np.argsort(y_prob[hr_indices])[::-1]]
 
dashboard_rows = []
for idx in hr_sorted:
    prob       = y_prob[idx]
    actual     = "Churned" if y_test[idx] else "Retained"
    cust_shap  = shap_df.iloc[idx]
    top2_feats = cust_shap[cust_shap > 0].nlargest(2)
 
    if len(top2_feats) == 0:
        top2_feats = cust_shap.nlargest(2)
 
    driver1 = top2_feats.index[0] if len(top2_feats) > 0 else "N/A"
    driver2 = top2_feats.index[1] if len(top2_feats) > 1 else "N/A"
    action  = ACTION_MAP.get(driver1, "Personal outreach — investigate pain points")
 
    # Pull readable raw values if available
    tenure_val   = X_test_raw.loc[idx, "tenure"] \
        if "tenure" in X_test_raw.columns else "N/A"
    charge_val   = X_test_raw.loc[idx, "MonthlyCharges"] \
        if "MonthlyCharges" in X_test_raw.columns else "N/A"
    contract_val = X_test_raw.loc[idx, "Contract"] \
        if "Contract" in X_test_raw.columns else "N/A"
 
    dashboard_rows.append({
        "Churn Prob"    : round(prob, 3),
        "Actual"        : actual,
        "Tenure (mo)"   : tenure_val,
        "Monthly (£)"   : charge_val,
        "Contract"      : contract_val,
        "Top Driver"    : driver1,
        "Driver 2"      : driver2,
        "Recommended Action": action,
    })
 
dashboard_df = pd.DataFrame(dashboard_rows)
dashboard_df.to_csv("churn_full_dashboard.csv", index=False)
 
print(f"\n{'='*65}")
print(f"INTERVENTION DASHBOARD — {len(dashboard_df)} HIGH-RISK CUSTOMERS")
print(f"{'='*65}")
print(dashboard_df.head(15)[["Churn Prob","Actual","Tenure (mo)",
                               "Monthly (£)","Contract",
                               "Top Driver","Recommended Action"]].to_string(index=False))
print(f"\n  Full dashboard saved: churn_full_dashboard.csv")